# PiShield — a Shield Layer for propositional requirements

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mihaela-stoian/PiShield/blob/hierarchical-requirements/examples/shield_layer_propositional.ipynb)

This notebook shows the two things a **Shield Layer** does for you on a small, visual 2D task:
**enforcement** and **delegation**. It reproduces the spirit of Figure 3 of CCN+ [2].

We have three classes: **A₁** and **A₂** are *subclasses* of **A**, and A is exactly their
union. This is captured by three propositional requirements — two positive and one with
**negation**:

- `A :- A₁` and `A :- A₂` — whenever A₁ or A₂ holds, A must hold too;
- `A₂ :- A, ¬A₁` — whenever A holds but A₁ does not, A₂ must hold.

- **Enforcement.** Wrapping the network in a Shield Layer makes its predictions *guaranteed*
  coherent: they satisfy all three rules — including the one with negation — no matter what the
  raw network says.
- **Delegation.** Because those classes are guaranteed to follow from the others, the network can
  *delegate* them rather than learn everything directly: A is derived as A₁ ∪ A₂, and A₂ can be
  derived as "A but not A₁" (delegation through negation). You'll see the raw network leave these
  to the shield, which recovers them exactly.

> Requires `torch` and `matplotlib` (both preinstalled on Colab).

## Setup

Uses a local PiShield checkout if you're running inside one; otherwise installs PiShield
(e.g. on a fresh Colab runtime). No datasets are downloaded — the task is generated in-notebook.

In [ ]:
import importlib.util, subprocess, sys, os

root = os.path.abspath(os.getcwd())
while root != os.path.dirname(root) and not os.path.isdir(os.path.join(root, 'pishield')):
    root = os.path.dirname(root)
if os.path.isdir(os.path.join(root, 'pishield')):
    sys.path.insert(0, root)
    print('Using local PiShield at', root)
elif importlib.util.find_spec('pishield') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'git+https://github.com/mihaela-stoian/PiShield.git@hierarchical-requirements'],
                   check=True)
    print('Installed PiShield')
else:
    print('PiShield already available')

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from pishield.shield_layer import build_shield_layer

torch.manual_seed(0)
np.random.seed(0)

## 1. A 2D task with a class hierarchy

Points live in the unit square. **A₂** is a big blob, **A₁** is a smaller blob elsewhere, and
**A** is their union (`A = A₁ ∪ A₂`). Each point gets three binary labels, in the order
`[A, A₁, A₂]`.

In [ ]:
def make_labels(P):
    x, y = P[:, 0], P[:, 1]
    in_A2 = ((x - 0.30) / 0.22) ** 2 + ((y - 0.40) / 0.26) ** 2 < 1   # big blob
    in_A1 = ((x - 0.62) / 0.13) ** 2 + ((y - 0.80) / 0.13) ** 2 < 1   # small blob
    in_A = in_A1 | in_A2                                              # A = A1 or A2
    return np.stack([in_A, in_A1, in_A2], 1).astype(np.float32)

N = 4000
X_train = np.random.rand(N, 2).astype(np.float32)
Y_train = make_labels(X_train)
X_t, Y_t = torch.tensor(X_train), torch.tensor(Y_train)
print('label counts [A, A1, A2]:', Y_train.sum(0).astype(int))

## 2. The requirements and the Shield Layer

With variables `A=0`, `A₁=1`, `A₂=2`, the three requirements are written as Horn rules:

- `0 :- 1` — `A :- A₁` (A holds if A₁ holds),
- `0 :- 2` — `A :- A₂` (A holds if A₂ holds),
- `2 :- 0 n1` — `A₂ :- A, ¬A₁` (A₂ holds if A holds and A₁ does **not**; the `n` prefix negates a literal).

We order the variables **A₁, A₂, A** (`custom_ordering='1,2,0'`); the layer uses the ordering to
decide how to correct and resolves the dependency between the rules automatically.

In [ ]:
constraints_path = 'shapes_constraints.txt'
with open(constraints_path, 'w') as f:
    f.write('0 :- 1\n')      # A  holds if A1 holds
    f.write('0 :- 2\n')      # A  holds if A2 holds
    f.write('2 :- 0 n1\n')   # A2 holds if A holds and A1 does NOT (negation)

shield = build_shield_layer(3, constraints_path,
                            ordering_choice='given', custom_ordering='1,2,0',
                            requirements_type='propositional')

## 3. A small classifier

A plain MLP mapping a 2D point to three per-class probabilities.

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2, 64), nn.ReLU(),
                                 nn.Linear(64, 64), nn.ReLU(),
                                 nn.Linear(64, 3))
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        return self.sigmoid(self.net(x))

## 4. Train through the Shield Layer

We apply the shield to the network's output (passing the labels as `goal`) and compute the loss
on the **shielded** output. Because A is guaranteed to follow from A₁/A₂, the loss on A is
satisfied without the network having to learn A itself — so it **delegates** A to the subclasses
and spends its capacity on A₁ and A₂.

In [ ]:
def train(epochs=400, lr=3e-3):
    model = MLP()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    bce = nn.BCELoss()
    for _ in range(epochs):
        opt.zero_grad()
        shielded = shield(model(X_t), goal=Y_t)   # enforce the requirements in the loss
        bce(shielded, Y_t).backward()
        opt.step()
    return model

model = train()
print('trained')

## 5. Enforcement: zero coherence violations

On a dense grid we count violations of **both** kinds of rule: the positive subclass rules
(`A₁ > 0.5` or `A₂ > 0.5` while `A ≤ 0.5`) and the **negation** rule (`A > 0.5` and `A₁ ≤ 0.5`
while `A₂ ≤ 0.5`). The raw network `f⁺` breaks them all over; the shielded output `CCN(h)`
never does — by construction.

In [ ]:
grid = np.linspace(0, 1, 140)
XX, YY = np.meshgrid(grid, grid)
G = torch.tensor(np.stack([XX.ravel(), YY.ravel()], 1).astype(np.float32))

model.eval()
with torch.no_grad():
    f_plus = model(G)          # raw network output
    ccn = shield(f_plus)       # shielded output (enforced)

def violations(out):
    A, A1, A2 = out[:, 0] > 0.5, out[:, 1] > 0.5, out[:, 2] > 0.5
    subclass = int((A1 & ~A).sum() + (A2 & ~A).sum())   # A1 -> A, A2 -> A
    negation = int((A & ~A1 & ~A2).sum())               # A and not A1 -> A2
    return subclass, negation

print('violations on the grid  (subclass rules, negation rule):')
print('  f+ (raw network): ', violations(f_plus))
print('  CCN(h) (shielded):', violations(ccn))

## 6. Decision boundaries

The top row is the raw network `f⁺`; the bottom row is the shielded `CCN(h)`. Blue = the model
is confident the point belongs to the class, red = it is not.

Look at the **Class A** column: `f⁺` barely predicts A (it delegated it), while `CCN(h)` shows A
as exactly the union of the A₁ and A₂ blobs — recovered by enforcement. The A₁ and A₂ columns
are unchanged, since the shield only had to raise A.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(9, 6.2))
names = ['A', 'A1', 'A2']
rows = [(f_plus, 'f+  (raw network)'), (ccn, 'CCN(h)  (shielded)')]
for i, (out, tag) in enumerate(rows):
    for j in range(3):
        ax = axes[i, j]
        cf = ax.contourf(XX, YY, out[:, j].reshape(XX.shape).numpy(),
                         levels=np.linspace(0, 1, 21), cmap='RdBu', vmin=0, vmax=1)
        ax.set_title(f'{tag}\nClass {names[j]}', fontsize=10)
        ax.set_xticks([]); ax.set_yticks([]); ax.set_aspect('equal')
fig.colorbar(cf, ax=axes, fraction=0.025, pad=0.02, label='predicted probability')
plt.show()

## 7. Takeaway

One Shield Layer gave us both halves of the story:

- **Enforcement** — the shielded predictions satisfy all three requirements everywhere, with zero
  violations — including the rule with **negation** (`A₂ :- A, ¬A₁`) — guaranteed for *any* input
  and *any* network.
- **Delegation** — the network does not have to learn every class directly. A is derived as
  A₁ ∪ A₂, and A₂ is derivable as "A but not A₁" — *delegation through negation*. This lets you
  state the requirements once and have the model focus its capacity on the parts it needs to learn.

The same recipe works for any propositional requirements — Horn rules `head :- body` (with
negated body literals via the `n` prefix) or disjunctive clauses like `y_0 or not y_1`.